In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os

# --- SETTINGS ---
INPUT_PKL = "../data/ukhls/pickles/o_indresp_backfilled.pkl"
OUTPUT_PKL = "../data/ukhls/pickles/normalized.pkl"

def create_normalized_pickle():
    if not os.path.exists(INPUT_PKL):
        print(f"Error: {INPUT_PKL} not found.")
        return

    print(f"Loading {INPUT_PKL}...")
    df = pd.read_pickle(INPUT_PKL)
    
    # We create a fresh dataframe for the normalized values to keep it clean
    df_norm = pd.DataFrame(index=df.index)
    df_norm['pidp'] = df['pidp']

    # 1. CONTINUOUS VARIABLES 
    continuous = [
        'o_age_dv', 'o_payn_dv', 'o_fimngrs_dv', 'o_sf12mcs_dv', 
        'o_sf12pcs_dv', 'o_jbttwt', 'o_carmiles', 'o_hhsize', 
        'o_nchild_dv', 'o_nbrsnci_dv'
    ]
    
    print("Processing continuous variables...")
    for col in continuous:
        if col in df.columns:
            # Force to numeric to bypass 'category' string issues
            numeric_series = pd.to_numeric(df[col], errors='coerce')
            
            # Fill missing income/commute distances with 0
            if col in ['o_payn_dv', 'o_fimngrs_dv', 'o_jbttwt', 'o_carmiles']:
                df_norm[col] = numeric_series.fillna(0)
                
                # Clip extreme UKHLS outliers that destroy K-Means centroids
                if col == 'o_carmiles':
                    df_norm[col] = df_norm[col].clip(upper=40000) # Reasonable max miles
                elif col == 'o_jbttwt':
                    df_norm[col] = df_norm[col].clip(upper=180) # Max 3hr commute one-way
            else:
                # Fill pure demographic missing data with median
                df_norm[col] = numeric_series.fillna(numeric_series.median())

    # 2. ORDINAL & BINARY MAPPINGS
    # We map these to specific floats, which automatically handles the category issue
    print("Mapping ordinals and binaries...")
    
    # Service Standard (1=Exc -> 5=Poor) maps to (1.0=Exc -> 0.0=Poor)
    service_map = {1: 1.0, 2: 0.75, 3: 0.5, 4: 0.25, 5: 0.0}
    service_cols = ['o_locserd', 'o_locserc', 'o_locsere', 'o_locsera']
    
    for col in service_cols:
        if col in df.columns:
            df_norm[col] = df[col].map(service_map).fillna(0.5).astype(float)

    # Education (1=Degree -> 9=None)
    if 'o_hiqual_dv' in df.columns:
        edu_map = {1: 1.0, 2: 0.8, 3: 0.6, 4: 0.4, 5: 0.2, 9: 0.0}
        df_norm['o_hiqual_dv'] = df['o_hiqual_dv'].map(edu_map).fillna(0.0).astype(float)

    # Binaries (1=Yes, 2=No)
    binaries = {
        'o_urban_dv': {1: 1.0, 2: 0.0},
        'o_englang': {1: 1.0, 2: 0.0},
        'o_sex_dv': {1: 1.0, 2: 0.0}
    }
    
    for col, mapping in binaries.items():
        if col in df.columns:
            df_norm[col] = df[col].map(mapping).fillna(0.0).astype(float)

    # 3. FINAL STANDARD SCALING (Z-SCORE)
    # This prevents extreme outliers in Income/Miles from crushing the significance of Age and Health Scores
    print("Applying Standard Scaling (Z-score)...")
    scaler = StandardScaler()
    # Scale everything except pidp
    cols_to_scale = [c for c in df_norm.columns if c != 'pidp']
    df_norm[cols_to_scale] = scaler.fit_transform(df_norm[cols_to_scale])

    # 4. SAVE
    df_norm.to_pickle(OUTPUT_PKL)
    print(f"Success! Normalized float matrix saved to {OUTPUT_PKL}")
    return df_norm

if __name__ == "__main__":
    df_normalized = create_normalized_pickle()

Loading ../data/ukhls/pickles/o_indresp_backfilled.pkl...
Processing continuous variables...
Mapping ordinals and binaries...
Applying Standard Scaling (Z-score)...
Success! Normalized float matrix saved to ../data/ukhls/pickles/normalized.pkl
